<a href="https://colab.research.google.com/github/shykeys-data/olist-ecommerce-bi/blob/main/notebooks/01_data_pipeline_star_schema.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Olist E-Commerce BI — Week 1: Data Pipeline & Star Schema

**Author:** Oluwaseyi ([@shykeys-data](https://github.com/shykeys-data))
**Dataset:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce): about 100k real orders from 2016–2018 in 9 linked tables
**Tools:** Python · pandas · DuckDB (SQL) · Plotly · Google Colab

---

### What this notebook does
Raw data is almost never ready for analysis. This notebook turns **9 messy, linked CSV files** into a **clean star schema**, the same kind of model that sits behind Power BI and Tableau dashboards at real companies.

| Step | What happens | Why it matters |
|---|---|---|
| 1. Setup | Install and import libraries | Anyone can re-run this notebook |
| 2. Get data | Download from Kaggle with an API key | No manual downloads, fully reproducible |
| 3. Profile | Check size, missing values and duplicates for every table | You can't clean what you haven't measured |
| 4. Clean | Fix dates, translate categories, remove duplicates, fix bad coordinates | Garbage in = garbage out |
| 5. Model | Build a star schema: 2 fact tables + 4 dimension tables | Fast, simple, consistent analysis |
| 6. Validate | Automatic checks that no data was lost or duplicated | Trust in the numbers |
| 7. First insight | SQL query + chart of monthly revenue | A first look at the business |

> **How to run:** add your Kaggle keys to Colab **Secrets** (🔑 icon on the left) as `KAGGLE_API_TOKEN` (or `KAGGLE_USERNAME` + `KAGGLE_KEY` if you have the older kaggle.json), then click **Runtime → Run all**.

## 1. Setup

In [ ]:
# Install the two libraries Colab may not have (takes ~10 seconds)
!pip install -q -U duckdb kagglehub kaggle

import os, glob
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("pandas", pd.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 5.2 MB/s eta 0:00:00
pandas 2.2.3


## 2. Get the data from Kaggle
The notebook reads your Kaggle keys from Colab **Secrets**, so they never appear in the code and never end up on GitHub.

In [ ]:
DATA_DIR = os.environ.get("OLIST_DATA_DIR")  # optional: point to a local folder instead of downloading

if DATA_DIR is None:
    from google.colab import userdata

    def get_secret(name):
        try:
            return userdata.get(name)
        except Exception:
            return None

    # Works with the classic kaggle.json keys (username + key) or the newer single API token
    if get_secret("KAGGLE_USERNAME") and get_secret("KAGGLE_KEY"):
        os.environ["KAGGLE_USERNAME"] = get_secret("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = get_secret("KAGGLE_KEY")
    token = get_secret("KAGGLE_API_TOKEN")
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token
        # Also save it where the Kaggle tools look for it automatically
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
            f.write(token)
        os.chmod(os.path.expanduser("~/.kaggle/access_token"), 0o600)

    try:
        import kagglehub
        DATA_DIR = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
    except Exception as e:
        # Fallback: the official Kaggle command-line tool
        print("kagglehub didn't work, trying the Kaggle CLI…", e)
        !kaggle datasets download -d olistbr/brazilian-ecommerce -p /content/olist --unzip -q
        DATA_DIR = "/content/olist"

print("Data folder:", DATA_DIR)
for f in sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True)):
    print(f"  {os.path.basename(f):45s} {os.path.getsize(f)/1e6:6.1f} MB")

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Data folder: /kaggle/input/brazilian-ecommerce
  olist_customers_dataset.csv                      9.0 MB
  olist_geolocation_dataset.csv                   61.3 MB
  olist_order_items_dataset.csv                   15.4 MB
  olist_order_payments_dataset.csv                 5.8 MB
  olist_order_reviews_dataset.csv                 14.5 MB
  olist_orders_dataset.csv                        17.7 MB
  olist_products_dataset.csv                       2.4 MB
  olist_sellers_dataset.csv                        0.2 MB
  product_category_name_translation.csv            0.0 MB


### Load all 9 tables
`utf-8-sig` handles a hidden character (BOM) at the start of one of the files that would otherwise break a column name.

In [ ]:
FILES = {
    "orders":      "olist_orders_dataset.csv",
    "items":       "olist_order_items_dataset.csv",
    "payments":    "olist_order_payments_dataset.csv",
    "reviews":     "olist_order_reviews_dataset.csv",
    "customers":   "olist_customers_dataset.csv",
    "sellers":     "olist_sellers_dataset.csv",
    "products":    "olist_products_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "translation": "product_category_name_translation.csv",
}

def find_file(name):
    hits = glob.glob(os.path.join(DATA_DIR, "**", name), recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name} not found in {DATA_DIR}")
    return hits[0]

raw = {key: pd.read_csv(find_file(fname), encoding="utf-8-sig") for key, fname in FILES.items()}

for key, df in raw.items():
    print(f"{key:12s} {df.shape[0]:>9,} rows × {df.shape[1]:>2} columns")

orders          99,441 rows ×  8 columns
items          112,650 rows ×  7 columns
payments       103,886 rows ×  5 columns
reviews         99,224 rows ×  7 columns
customers       99,441 rows ×  5 columns
sellers          3,095 rows ×  4 columns
products        32,951 rows ×  9 columns
geolocation  1,000,163 rows ×  5 columns
translation         71 rows ×  2 columns


## 3. Profile the raw data
Before cleaning anything, measure it. For every table: row count, missing values, fully duplicated rows, and whether the **key** that should identify each row is actually unique.

In [ ]:
# The column (or columns) that SHOULD uniquely identify a row in each table
EXPECTED_KEYS = {
    "orders": ["order_id"],
    "items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["review_id"],
    "customers": ["customer_id"],
    "sellers": ["seller_id"],
    "products": ["product_id"],
    "geolocation": ["geolocation_zip_code_prefix"],
    "translation": ["product_category_name"],
}

def profile(name, df):
    key = EXPECTED_KEYS[name]
    return {
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "missing_cells": int(df.isna().sum().sum()),
        "missing_%": round(100 * df.isna().sum().sum() / df.size, 2),
        "duplicate_rows": int(df.duplicated().sum()),
        "key": " + ".join(key),
        "key_is_unique": not df.duplicated(subset=key).any(),
    }

profile_df = pd.DataFrame([profile(n, d) for n, d in raw.items()])
profile_df

,table,rows,columns,missing_cells,missing_%,duplicate_rows,key,key_is_unique
0,orders,99441,8,4908,0.62,0,order_id,True
1,items,112650,7,0,0.00,0,order_id + order_item_id,True
2,payments,103886,5,0,0.00,0,order_id + payment_sequential,True
3,reviews,99224,7,145903,21.01,0,review_id,False
4,customers,99441,5,0,0.00,0,customer_id,True
5,sellers,3095,4,0,0.00,0,seller_id,True
6,products,32951,9,2448,0.83,0,product_id,True
7,geolocation,1000163,5,0,0.00,261831,geolocation_zip_code_prefix,False
8,translation,71,2,0,0.00,0,product_category_name,True


In [ ]:
# Which columns have missing values, and how many?
missing = (
    pd.concat({n: d.isna().sum() for n, d in raw.items()})
      .rename("missing").reset_index()
      .rename(columns={"level_0": "table", "level_1": "column"})
)
missing = missing[missing["missing"] > 0].sort_values("missing", ascending=False)
missing["missing_%"] = missing.apply(lambda r: 100 * r["missing"] / len(raw[r["table"]]), axis=1).round(2)
missing

,table,column,missing,missing_%
23,reviews,review_comment_title,87656,88.34
24,reviews,review_comment_message,58247,58.70
6,orders,order_delivered_customer_date,2965,2.98
5,orders,order_delivered_carrier_date,1783,1.79
37,products,product_category_name,610,1.85
39,products,product_description_lenght,610,1.85
38,products,product_name_lenght,610,1.85
40,products,product_photos_qty,610,1.85
4,orders,order_approved_at,160,0.16
41,products,product_weight_g,2,0.01


### What the profile tells us (and how each issue is handled)

| Issue found | Why it happens | Fix |
|---|---|---|
| `reviews`: `review_id` is not unique, and some orders have more than one review | Customers can review again and some reviews are copied | Keep **one review per order**: the latest |
| `geolocation`: many rows per zip code, some coordinates outside Brazil | Each zip area has many GPS points, plus some entry errors | Keep only points inside Brazil, then **average per zip code** |
| `orders`: missing delivery dates | Orders that were canceled or never delivered | Keep them, but mark delivery fields as not applicable |
| `products`: missing category | Missing info from the seller | Label as `unknown` |
| `customers`: `customer_id` changes with every order | Olist creates a new id per order | Use `customer_unique_id` as the real customer |

That last one matters: if you count `customer_id`, almost every customer looks like a one-time buyer. Getting this wrong would ruin any repeat-purchase or retention analysis later.

## 4. Clean each table

### 4.1 Orders: convert dates and calculate delivery metrics

In [ ]:
orders = raw["orders"].copy()

date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")

# Sanity check: a delivery cannot happen before the purchase
bad_delivery = orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]
print("Deliveries dated before purchase:", int(bad_delivery.sum()))
orders.loc[bad_delivery, "order_delivered_customer_date"] = pd.NaT

# Delivery metrics (only meaningful when the order was actually delivered)
delivered_mask = orders["order_delivered_customer_date"].notna()
orders["delivery_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.total_seconds() / 86400
orders["promised_days"] = (orders["order_estimated_delivery_date"] - orders["order_purchase_timestamp"]).dt.total_seconds() / 86400
orders["days_late"] = (orders["order_delivered_customer_date"].dt.normalize() - orders["order_estimated_delivery_date"].dt.normalize()).dt.days
orders["is_late"] = (orders["days_late"] > 0).astype("boolean").where(delivered_mask)  # empty if not delivered

print("Order statuses:")
print(orders["order_status"].value_counts().to_string())
print(f"\nPurchase dates: {orders['order_purchase_timestamp'].min():%Y-%m-%d} → {orders['order_purchase_timestamp'].max():%Y-%m-%d}")

Deliveries dated before purchase: 0
Order statuses:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2

Purchase dates: 2016-09-04 → 2018-10-17


### 4.2 Order items: dates and line totals

In [ ]:
items = raw["items"].copy()
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"], errors="coerce")
items["item_total"] = items["price"] + items["freight_value"]   # what the customer paid for this line
print(items[["price", "freight_value", "item_total"]].describe().T)

                   count   mean    std  min   25%   50%    75%      max
price         112,650.00 120.65 183.63 0.85 39.90 74.99 134.90 6,735.00
freight_value 112,650.00  19.99  15.81 0.00 13.08 16.26  21.15   409.68
item_total    112,650.00 140.64 190.72 6.08 55.22 92.32 157.94 6,929.31


### 4.3 Payments: one row per order
An order can be paid in several parts (for example two cards plus a voucher). For modelling we need **one row per order**: the total paid, the number of payments, the main payment method, and the number of instalments.

In [ ]:
payments = raw["payments"].copy()
before = len(payments)
payments = payments[~((payments["payment_type"] == "not_defined") & (payments["payment_value"] == 0))]
print("Removed empty 'not_defined' payments:", before - len(payments))

main_type = (payments.sort_values("payment_value", ascending=False)
                     .drop_duplicates("order_id")[["order_id", "payment_type"]]
                     .rename(columns={"payment_type": "main_payment_type"}))

payments_by_order = (payments.groupby("order_id")
                     .agg(total_paid=("payment_value", "sum"),
                          n_payments=("payment_sequential", "count"),
                          installments=("payment_installments", "max"))
                     .reset_index()
                     .merge(main_type, on="order_id", how="left"))
payments_by_order.head()

Removed empty 'not_defined' payments: 3


,order_id,total_paid,n_payments,installments,main_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,2,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3,credit_card


### 4.4 Reviews: one review per order (the latest)

In [ ]:
reviews = raw["reviews"].copy()
for c in ["review_creation_date", "review_answer_timestamp"]:
    reviews[c] = pd.to_datetime(reviews[c], errors="coerce")

print("Exact duplicate rows:", int(reviews.duplicated().sum()))
print("Orders with more than one review:", int(reviews["order_id"].duplicated().sum()))

reviews_by_order = (reviews.sort_values("review_answer_timestamp")
                           .drop_duplicates("order_id", keep="last")
                           [["order_id", "review_score", "review_comment_message", "review_creation_date"]])
reviews_by_order["has_comment"] = reviews_by_order["review_comment_message"].notna()
print("Reviews kept:", len(reviews_by_order))

Exact duplicate rows: 0
Orders with more than one review: 551
Reviews kept: 98673


### 4.5 Products: English categories and tidy column names

In [ ]:
products = raw["products"].copy()

# The original file misspells 'length' as 'lenght'
products = products.rename(columns={"product_name_lenght": "product_name_length",
                                    "product_description_lenght": "product_description_length"})

translation = dict(zip(raw["translation"]["product_category_name"],
                       raw["translation"]["product_category_name_english"]))
# Two categories are missing from the official translation file
translation.setdefault("pc_gamer", "pc_gamer")
translation.setdefault("portateis_cozinha_e_preparadores_de_alimentos", "portable_kitchen_and_food_preparers")

products["product_category_name"] = products["product_category_name"].fillna("unknown")
products["category"] = products["product_category_name"].map(translation).fillna(products["product_category_name"])
products["category"] = products["category"].str.replace("_", " ").str.title()

print("Products:", len(products), "| Categories:", products["category"].nunique())
print("Products with unknown category:", int((products["category"] == "Unknown").sum()))

Products: 32951 | Categories: 74
Products with unknown category: 610


### 4.6 Geolocation: one coordinate per zip code
Brazil sits roughly between latitude −34 and +6 and longitude −74 and −34. Points outside that box are entry errors.

In [ ]:
geo = raw["geolocation"].copy()
in_brazil = geo["geolocation_lat"].between(-34.0, 5.5) & geo["geolocation_lng"].between(-74.0, -34.0)
print("Rows outside Brazil removed:", int((~in_brazil).sum()))

geo_by_zip = (geo[in_brazil]
              .groupby("geolocation_zip_code_prefix")
              .agg(lat=("geolocation_lat", "mean"), lng=("geolocation_lng", "mean"))
              .reset_index()
              .rename(columns={"geolocation_zip_code_prefix": "zip_code_prefix"}))
print("Zip codes with coordinates:", len(geo_by_zip))

Rows outside Brazil removed: 42
Zip codes with coordinates: 19010


## 5. Build the star schema

A **star schema** puts the numbers you measure (sales, delivery days, review scores) in **fact tables** in the middle, surrounded by **dimension tables** that describe them (who, what, where, when).

```
                 dim_date
                    │
 dim_customer ── fact_orders ── fact_order_items ── dim_product
                                       │
                                   dim_seller
```

| Table | One row = | Main columns |
|---|---|---|
| `fact_order_items` | one product line in an order | price, freight, item total |
| `fact_orders` | one order | order value, amount paid, delivery days, late flag, review score |
| `dim_customer` | one real customer (`customer_unique_id`) | city, state, location, first purchase date |
| `dim_product` | one product | category, weight, size |
| `dim_seller` | one seller | city, state, location |
| `dim_date` | one calendar day | year, quarter, month, weekday |

**Why two fact tables?** Revenue lives at item level, while delivery and reviews live at order level. Forcing them into one table would repeat each order's review score once per item and skew the averages. Picking the right level of detail ("grain") for each table is a core data-modelling skill.

### 5.1 dim_date

In [ ]:
start = orders["order_purchase_timestamp"].min().normalize()
end = orders["order_estimated_delivery_date"].max().normalize()
dates = pd.date_range(start, end, freq="D")

dim_date = pd.DataFrame({
    "date_key": dates.strftime("%Y%m%d").astype(int),
    "date": dates,
    "year": dates.year,
    "quarter": dates.quarter,
    "month": dates.month,
    "month_name": dates.strftime("%b"),
    "year_month": dates.strftime("%Y-%m"),
    "week_of_year": dates.isocalendar().week.astype(int).values,
    "day_of_week": dates.strftime("%a"),
    "is_weekend": dates.dayofweek >= 5,
})
print(len(dim_date), "days")
dim_date.head(3)

800 days


,date_key,date,year,quarter,month,month_name,year_month,week_of_year,day_of_week,is_weekend
0,20160904,2016-09-04,2016,3,9,Sep,2016-09,35,Sun,True
1,20160905,2016-09-05,2016,3,9,Sep,2016-09,36,Mon,False
2,20160906,2016-09-06,2016,3,9,Sep,2016-09,36,Tue,False


### 5.2 dim_customer
Each real customer gets their location from their **most recent** order, plus their **first purchase date**, which we'll need for cohort analysis in Week 2.

In [ ]:
cust = raw["customers"].merge(orders[["customer_id", "order_purchase_timestamp"]], on="customer_id", how="left")

latest = (cust.sort_values("order_purchase_timestamp")
              .drop_duplicates("customer_unique_id", keep="last"))
first_purchase = cust.groupby("customer_unique_id")["order_purchase_timestamp"].min().rename("first_purchase_date")

dim_customer = (latest[["customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state"]]
                .merge(first_purchase, on="customer_unique_id", how="left")
                .merge(geo_by_zip, left_on="customer_zip_code_prefix", right_on="zip_code_prefix", how="left")
                .drop(columns="zip_code_prefix")
                .rename(columns={"customer_zip_code_prefix": "zip_code_prefix",
                                 "customer_city": "city", "customer_state": "state"}))
dim_customer["city"] = dim_customer["city"].str.title()

print(f"customer_id values: {raw['customers']['customer_id'].nunique():,}  →  real customers: {len(dim_customer):,}")
dim_customer.head(3)

customer_id values: 99,441  →  real customers: 96,096


,customer_unique_id,zip_code_prefix,city,state,first_purchase_date,lat,lng
0,b7d76e111c89f7ebf14761390f0f7d17,69309,Boa Vista,RR,2016-09-04 21:15:19,2.81,-60.70
1,4854e9b3feff728c13ee5fc7d1547e92,99025,Passo Fundo,RS,2016-09-05 00:15:34,-28.26,-52.43
2,009b0127b727ab0ba422f6d9604487c7,12244,Sao Jose Dos Campos,SP,2016-09-13 15:24:19,-23.20,-45.94


### 5.3 dim_product and dim_seller

In [ ]:
dim_product = products[["product_id", "category", "product_category_name", "product_photos_qty",
                        "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]] \
                .rename(columns={"product_category_name": "category_pt"})

dim_seller = (raw["sellers"]
              .merge(geo_by_zip, left_on="seller_zip_code_prefix", right_on="zip_code_prefix", how="left")
              .drop(columns="zip_code_prefix")
              .rename(columns={"seller_zip_code_prefix": "zip_code_prefix",
                               "seller_city": "city", "seller_state": "state"}))
dim_seller["city"] = dim_seller["city"].str.title()

print("dim_product:", len(dim_product), "| dim_seller:", len(dim_seller))

dim_product: 32951 | dim_seller: 3095


### 5.4 fact_orders (one row per order)

In [ ]:
items_by_order = (items.groupby("order_id")
                  .agg(n_items=("order_item_id", "count"),
                       n_sellers=("seller_id", "nunique"),
                       items_value=("price", "sum"),
                       freight_value=("freight_value", "sum"))
                  .reset_index())

fact_orders = (orders
    .merge(raw["customers"][["customer_id", "customer_unique_id"]], on="customer_id", how="left")
    .merge(items_by_order, on="order_id", how="left")
    .merge(payments_by_order, on="order_id", how="left")
    .merge(reviews_by_order[["order_id", "review_score", "has_comment"]], on="order_id", how="left"))

fact_orders["purchase_date_key"] = fact_orders["order_purchase_timestamp"].dt.strftime("%Y%m%d").astype(int)
fact_orders["n_items"] = fact_orders["n_items"].fillna(0).astype(int)
fact_orders["order_value"] = fact_orders["items_value"] + fact_orders["freight_value"]

fact_orders = fact_orders[[
    "order_id", "customer_unique_id", "purchase_date_key", "order_status",
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
    "n_items", "n_sellers", "items_value", "freight_value", "order_value",
    "total_paid", "n_payments", "main_payment_type", "installments",
    "delivery_days", "promised_days", "days_late", "is_late",
    "review_score", "has_comment",
]]
fact_orders["has_comment"] = fact_orders["has_comment"].astype("boolean")  # True / False / empty (no review)
print(f"fact_orders: {len(fact_orders):,} rows")
fact_orders.head(3)

fact_orders: 99,441 rows


,order_id,customer_unique_id,purchase_date_key,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,n_items,n_sellers,items_value,freight_value,order_value,total_paid,n_payments,main_payment_type,installments,delivery_days,promised_days,days_late,is_late,review_score,has_comment
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,20171002,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,1.00,29.99,8.72,38.71,38.71,3.00,voucher,1.00,8.44,15.54,-8.00,False,4.00,True
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,20180724,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1,1.00,118.70,22.76,141.46,141.46,1.00,boleto,1.00,13.78,19.14,-6.00,False,4.00,True
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,20180808,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1,1.00,159.90,19.22,179.12,179.12,1.00,credit_card,3.00,9.39,26.64,-18.00,False,5.00,False


### 5.5 fact_order_items (one row per product line)

In [ ]:
fact_order_items = (items
    .merge(fact_orders[["order_id", "customer_unique_id", "purchase_date_key", "order_status"]],
           on="order_id", how="left")
    [["order_id", "order_item_id", "product_id", "seller_id", "customer_unique_id",
      "purchase_date_key", "order_status", "shipping_limit_date",
      "price", "freight_value", "item_total"]])
print(f"fact_order_items: {len(fact_order_items):,} rows")
fact_order_items.head(3)

fact_order_items: 112,650 rows


,order_id,order_item_id,product_id,seller_id,customer_unique_id,purchase_date_key,order_status,shipping_limit_date,price,freight_value,item_total
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,871766c5855e863f6eccc05f988b23cb,20170913,delivered,2017-09-19 09:45:35,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,eb28e67c4c0b83846050ddfb8a35d051,20170426,delivered,2017-05-03 11:05:13,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,3818d81c6709e39d06b2738a8d3a2474,20180114,delivered,2018-01-18 14:48:30,199.00,17.87,216.87


## 6. Validate the model
These checks run automatically. If any number were lost or duplicated during the joins, the notebook stops with an error instead of quietly producing wrong results.

In [ ]:
checks = []
def check(name, passed, detail=""):
    checks.append({"check": name, "result": "✅ PASS" if passed else "❌ FAIL", "detail": detail})

# 1. Each table has exactly one row per key (its 'grain')
for name, df, key in [("fact_orders", fact_orders, ["order_id"]),
                      ("fact_order_items", fact_order_items, ["order_id", "order_item_id"]),
                      ("dim_customer", dim_customer, ["customer_unique_id"]),
                      ("dim_product", dim_product, ["product_id"]),
                      ("dim_seller", dim_seller, ["seller_id"]),
                      ("dim_date", dim_date, ["date_key"])]:
    dups = int(df.duplicated(subset=key).sum())
    check(f"{name}: one row per {' + '.join(key)}", dups == 0, f"{dups} duplicates")

# 2. No rows were lost or added
check("All orders kept", len(fact_orders) == len(raw["orders"]), f"{len(fact_orders):,} vs {len(raw['orders']):,}")
check("All order items kept", len(fact_order_items) == len(raw["items"]), f"{len(fact_order_items):,} vs {len(raw['items']):,}")

# 3. Money adds up: revenue in the model = revenue in the raw files
raw_rev = (raw["items"]["price"] + raw["items"]["freight_value"]).sum()
model_rev = fact_order_items["item_total"].sum()
check("Item revenue matches raw data", np.isclose(raw_rev, model_rev), f"R$ {model_rev:,.2f}")
check("Order-level revenue = item-level revenue", np.isclose(fact_orders["order_value"].sum(), model_rev))

# 4. Every key in the facts exists in its dimension
def fk(fact, col, dim, dim_col):
    missing = int((~fact[col].isin(dim[dim_col])).sum())
    check(f"{col} found in {dim_col.split('_')[0]} dimension", missing == 0, f"{missing} missing")
fk(fact_orders, "customer_unique_id", dim_customer, "customer_unique_id")
fk(fact_orders, "purchase_date_key", dim_date, "date_key")
fk(fact_order_items, "product_id", dim_product, "product_id")
fk(fact_order_items, "seller_id", dim_seller, "seller_id")

checks_df = pd.DataFrame(checks)
try:
    display(checks_df)          # nice table in Colab
except NameError:
    print(checks_df.to_string())
assert (checks_df["result"] == "✅ PASS").all(), "Some checks failed: see table above"
print("\nAll checks passed ✅")

,check,result,detail
0,fact_orders: one row per order_id,✅ PASS,0 duplicates
1,fact_order_items: one row per order_id + order...,✅ PASS,0 duplicates
2,dim_customer: one row per customer_unique_id,✅ PASS,0 duplicates
3,dim_product: one row per product_id,✅ PASS,0 duplicates
4,dim_seller: one row per seller_id,✅ PASS,0 duplicates
5,dim_date: one row per date_key,✅ PASS,0 duplicates
6,All orders kept,✅ PASS,"99,441 vs 99,441"
7,All order items kept,✅ PASS,"112,650 vs 112,650"
8,Item revenue matches raw data,✅ PASS,"R$ 15,843,553.24"
9,Order-level revenue = item-level revenue,✅ PASS,



All checks passed ✅


In [ ]:
# Informational: does what customers paid match the order value?
# Small gaps are normal (instalment interest, vouchers), so this is a note rather than a failing check
diff = (fact_orders["total_paid"] - fact_orders["order_value"]).abs()
has_both = fact_orders["total_paid"].notna() & fact_orders["order_value"].notna()
print(f"Orders where payment differs from order value by more than R$1: "
      f"{(diff[has_both] > 1).mean():.1%}")

Orders where payment differs from order value by more than R$1: 0.3%


## 7. First insight: monthly revenue (with SQL)
The model is ready, so now we can query it with **SQL**, the same way an analyst queries a company data warehouse. DuckDB runs SQL directly on our pandas tables.

**Revenue definition (GMV):** product price + freight for **delivered** orders. Writing the definition down matters: "revenue" means different things to different teams.

In [ ]:
import duckdb

monthly = duckdb.sql('''
    SELECT d.year_month,
           COUNT(DISTINCT f.order_id)          AS orders,
           ROUND(SUM(f.order_value), 2)        AS revenue,
           ROUND(AVG(f.order_value), 2)        AS avg_order_value,
           ROUND(AVG(f.review_score), 2)       AS avg_review
    FROM fact_orders f
    JOIN dim_date d ON f.purchase_date_key = d.date_key
    WHERE f.order_status = 'delivered'
    GROUP BY d.year_month
    ORDER BY d.year_month
''').df()

# The first and last few months only have a handful of orders (the platform was starting up
# and data collection was ending), so we focus on the complete period
monthly_full = monthly[(monthly["year_month"] >= "2017-01") & (monthly["year_month"] <= "2018-08")]
monthly_full

,year_month,orders,revenue,avg_order_value,avg_review
3,2017-01,750,"127,482.37",169.98,4.20
4,2017-02,1653,"271,239.32",164.09,4.20
5,2017-03,2546,"414,330.95",162.74,4.19
6,2017-04,2303,"390,812.40",169.70,4.14
7,2017-05,3546,"566,851.40",159.86,4.24
8,2017-06,3135,"490,050.37",156.32,4.22
9,2017-07,3872,"566,299.08",146.25,4.26
10,2017-08,4193,"645,832.36",154.03,4.31
11,2017-09,4150,"701,077.49",168.93,4.27
12,2017-10,4478,"751,117.01",167.73,4.20


In [ ]:
import plotly.express as px

fig = px.bar(monthly_full, x="year_month", y="revenue",
             title="Olist monthly revenue (delivered orders, Jan 2017 – Aug 2018)",
             labels={"year_month": "", "revenue": "Revenue (R$)"},
             hover_data={"orders": ":,", "avg_order_value": ":.2f"})
fig.update_traces(marker_color="#2E6FDB")
fig.update_layout(template="simple_white", yaxis_tickformat=",.0f", height=420)
fig.show()

### Headline numbers

In [ ]:
delivered = fact_orders[fact_orders["order_status"] == "delivered"]
repeat_customers = (fact_orders.groupby("customer_unique_id")["order_id"].nunique() > 1).mean()

headline = pd.Series({
    "Orders": f"{len(fact_orders):,}",
    "Real customers": f"{len(dim_customer):,}",
    "Sellers": f"{len(dim_seller):,}",
    "Products": f"{len(dim_product):,}",
    "Revenue, delivered orders (R$)": f"{delivered['order_value'].sum():,.0f}",
    "Average order value (R$)": f"{delivered['order_value'].mean():,.2f}",
    "Delivered late": f"{delivered['is_late'].mean():.1%}",
    "Average delivery time (days)": f"{delivered['delivery_days'].mean():.1f}",
    "Average review score (1–5)": f"{delivered['review_score'].mean():.2f}",
    "Customers who ordered more than once": f"{repeat_customers:.1%}",
}, name="value")
headline.to_frame()

,value
Orders,"99,441"
Real customers,"96,096"
Sellers,"3,095"
Products,"32,951"
"Revenue, delivered orders (R$)","15,419,774"
Average order value (R$),159.83
Delivered late,6.8%
Average delivery time (days),12.6
Average review score (1–5),4.16
Customers who ordered more than once,3.1%


## 8. Save the model
The six tables are saved as **Parquet** files: a compressed, typed format that loads much faster than CSV. Next week's notebook starts from these files.

> Colab deletes files when the session ends. Run the optional Google Drive cell to keep a copy.

In [ ]:
OUT_DIR = "model"
os.makedirs(OUT_DIR, exist_ok=True)

tables = {"fact_orders": fact_orders, "fact_order_items": fact_order_items,
          "dim_customer": dim_customer, "dim_product": dim_product,
          "dim_seller": dim_seller, "dim_date": dim_date}

for name, df in tables.items():
    try:
        df.to_parquet(f"{OUT_DIR}/{name}.parquet", index=False)
        ext = "parquet"
    except ImportError:          # fallback if pyarrow isn't installed
        df.to_csv(f"{OUT_DIR}/{name}.csv", index=False)
        ext = "csv"
    print(f"saved {name:18s} {len(df):>8,} rows  →  {OUT_DIR}/{name}.{ext}")

saved fact_orders          99,441 rows  →  model/fact_orders.parquet
saved fact_order_items    112,650 rows  →  model/fact_order_items.parquet
saved dim_customer         96,096 rows  →  model/dim_customer.parquet
saved dim_product          32,951 rows  →  model/dim_product.parquet
saved dim_seller            3,095 rows  →  model/dim_seller.parquet
saved dim_date                800 rows  →  model/dim_date.parquet


In [ ]:
# OPTIONAL: copy the model to Google Drive so it survives after Colab closes
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p /content/drive/MyDrive/olist-ecommerce-bi && cp -r model /content/drive/MyDrive/olist-ecommerce-bi/

---
## Summary: what I did and why

1. **Downloaded** the data automatically through the Kaggle API, with keys kept out of the code.
2. **Profiled** all 9 tables and found the real problems: duplicate reviews, GPS points outside Brazil, missing categories, misspelled column names, and a customer ID that changes with every order.
3. **Cleaned** each problem with a documented rule instead of deleting data quietly.
4. **Modelled** the data as a star schema with two fact tables at different grains (order and order item) and four dimensions.
5. **Validated** the model automatically: no duplicates, no lost rows, revenue reconciles to the cent, and every key matches.
6. **Queried** the model with SQL to get a first view of monthly revenue.

### Next: Week 2
KPI layer · customer cohort retention · RFM segmentation · revenue by region and category · **how late delivery affects review scores**